# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema and available here: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In this notebook we reference **all record sets, fields, and columns using their `@id` values** as defined in the Croissant schema.

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and initialize the dataset with `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Set Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
ds = mlc.Dataset(croissant_url)
# Get dataset metadata (object, not subscriptable)
metadata = ds.metadata
print(f"Dataset: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
List all available record sets, fields, and columns in the dataset. Each has an `@id` property that is used for referencing.

**Tip:** The dataset's schema may be nested. Use `.metadata.record_sets` to list top-level record sets, and check their `fields` and `columns` for further structure.

In [ ]:
# Print record set @id, name, field, and column @id details
record_sets = getattr(metadata, 'record_sets', [])
for rs in record_sets:
    print(f"Record Set: {getattr(rs, '@id', 'N/A')}  Name: {getattr(rs, 'name', '')}")
    print("  Fields and Columns:")
    for field in getattr(rs, 'fields', []):
        print(f"    Field @id: {getattr(field, '@id', 'N/A'): <45}  Name: {getattr(field, 'name', 'N/A')}")
    for column in getattr(rs, 'columns', []):
        print(f"    Column @id: {getattr(column, '@id', 'N/A'): <44} Name: {getattr(column, 'name', 'N/A')}")
    print('-'*70)
# Select one main record set and fields for next steps.
# If the dataset does not define any record_sets explicitly, find available top-level attributeif not record_sets:
    print("No record sets found at 'metadata.record_sets'. Check the Croissant schema for structure or consult documentation.")

## 3. Data Extraction
Load the records for a chosen record set into a pandas DataFrame. All entity references use their Croissant `@id`.

> ⚠️ **Note:** Replace `<chosen_record_set_id>` with one of the record set `@id`s above. If the list is empty and the Croissant describes a single default record set, try using `record_set=None` or consult the schema.

In [ ]:
# For demonstration, auto-detect a record set or use None if missing
if record_sets and hasattr(record_sets[0], '@id'):
    chosen_record_set_id = getattr(record_sets[0], '@id')
    print(f"Using record set @id: {chosen_record_set_id}")
else:
    chosen_record_set_id = None
    print("Fallback: Using record_set=None (the only available logical table)")

all_record_set_ids = [getattr(rs, '@id') for rs in record_sets] if record_sets else [None]

dataframes = {}
for record_set_id in all_record_set_ids:
    records = list(ds.records(record_set=record_set_id))  # record_set=None gets the default
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} rows for record_set_id: {record_set_id}")

# Show available columns
main_df = dataframes[chosen_record_set_id]
print("Columns in main DataFrame:")
print(main_df.columns.tolist())
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Let's analyze a numeric field (e.g., `cr:field:mw` for molecular weight, or `cr:field:coverage_percent` / similar, depending on what is available).

> ⚠️ **Instructions:**
>
- Pick the exact `@id` of a numeric field (such as a protein attribute, or sample abundance) from the data above. You can get all field names from `main_df.columns`.
- Replace `<numeric_field_id>` and `<group_field_id>` accordingly.

In [ ]:
# Pick a numeric field
available_numeric_fields = [col for col in main_df.columns if pd.api.types.is_numeric_dtype(main_df[col])]
print(f"Available numeric fields: {available_numeric_fields}")

if available_numeric_fields:
    numeric_field = available_numeric_fields[0]   # You may set a preferred field by '@id'
else:
    # Try to guess some default columns by domain knowledge
    possible_numeric_fields = [
        'cr:field:mw',            # molecular weight
        'cr:field:coverage_percent',
        'cr:field:peptide_count'  # invented: check schema for real @id
    ]
    possible_numeric_field = None
    for k in main_df.columns:
        if k in possible_numeric_fields:
            possible_numeric_field = k
            break
    numeric_field = possible_numeric_field

if not numeric_field:
    print('No numeric field detected. Please set `numeric_field` to a valid column name as @id from previous lists.')

## FILTER EXAMPLE
if numeric_field and numeric_field in main_df.columns:
    threshold = main_df[numeric_field].quantile(0.9) if main_df[numeric_field].notnull().any() else 0
    filtered_df = main_df[main_df[numeric_field] > threshold]
    print(f"Filtered records where '{numeric_field}' > {threshold:.2f} (top 10%):")
    print(filtered_df[[numeric_field]].head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by another field (categorical ID), e.g., 'cr:field:sample_id', 'cr:field:description', etc.
    preferred_groups = [
        'cr:field:description',
        'cr:field:sample_id',     # invented: check the Croissant schema
        'cr:field:accession'
    ]
    group_field = None
    for g in main_df.columns:
        if g in preferred_groups:
            group_field = g
            break

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped mean {numeric_field} by {group_field}:")
        print(grouped_df.head())
    else:
        print("No suitable group field found. You may edit 'group_field' manually.")
else:
    print("Please edit the code to select a valid column as `numeric_field` (see above for choices, must be present in DataFrame columns).")

## 5. Visualization
Now, let's plot the distribution of a numeric variable for further inspection.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and numeric_field in main_df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(main_df[numeric_field].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # If there is a group field, plot per group
    if 'group_field' in locals() and group_field and group_field in main_df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=main_df, x=group_field, y=numeric_field)
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field} by {group_field}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook we explored the FAIR² mass spectrometry dataset describing extracellular vesicle proteins. We loaded the data directly from its Croissant schema, listed its structure using unique `@id` references, extracted and processed one primary record set, and visualized molecular features. This workflow can be extended to other bioinformatics record sets following FAIR principles with `mlcroissant`.

*Notebook created following best practices for reproducible and FAIR data exploration with croissant-backed datasets.*